# 02 — Global Codebase Profile
**Purpose:** Statistical portrait of the codebase — distributions, correlations,
segmentation, 80/20 analysis.
**Inputs:** `file_metrics.parquet`, `target_metrics.parquet`
**Outputs:** `data/intermediate/target_features.parquet`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from buildanalysis.loading import BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

# Style
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)
fm = ds.file_metrics
tm = ds.target_metrics

print(f"file_metrics:   {fm.shape[0]:,} rows x {fm.shape[1]} cols")
print(f"target_metrics: {tm.shape[0]:,} rows x {tm.shape[1]} cols")

/Users/cgilbert/Work/projects/build-optimisation/.venv/lib/python3.13/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


FileNotFoundError: ../data/processed/file_metrics.parquet not found. This file is produced by scripts/consolidate/build_file_metrics.py. Run the data collection pipeline first.

## 1. File-Level Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1a. Compile time — histogram + CDF
ct = fm["compile_time_ms"].dropna()
ax = axes[0, 0]
ax.hist(ct[ct > 0], bins=80, edgecolor="white", linewidth=0.3)
ax.set_xscale("log")
ax.set_xlabel("Compile time (ms)")
ax.set_ylabel("Count")
ax.set_title("Compile Time Distribution")
# CDF overlay on twin axis
ax2 = ax.twinx()
sorted_ct = np.sort(ct[ct > 0])
cdf = np.arange(1, len(sorted_ct) + 1) / len(sorted_ct)
ax2.plot(sorted_ct, cdf, color="red", linewidth=1.5, label="CDF")
ax2.set_ylabel("CDF", color="red")
ax2.legend(loc="center right")

# 1b. Code lines
ax = axes[0, 1]
cl = fm["code_lines"].dropna()
ax.hist(cl[cl > 0], bins=80, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Lines of code")
ax.set_ylabel("Count")
ax.set_title("Code Lines Distribution")

# 1c. Preprocessed bytes
ax = axes[1, 0]
pb = fm["preprocessed_bytes"].dropna()
ax.hist(pb[pb > 0], bins=80, edgecolor="white", linewidth=0.3, color="green")
ax.set_xscale("log")
ax.set_xlabel("Preprocessed bytes")
ax.set_ylabel("Count")
ax.set_title("Preprocessed Size Distribution")

# 1d. Expansion ratio
ax = axes[1, 1]
er = fm["expansion_ratio"].dropna()
ax.hist(er[er > 0], bins=80, edgecolor="white", linewidth=0.3, color="orange")
ax.set_xlabel("Expansion ratio (preprocessed / source)")
ax.set_ylabel("Count")
ax.set_title("Expansion Ratio Distribution")

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: code_lines vs compile_time_ms (colored by is_generated)
ax = axes[0]
plot_df = fm[["code_lines", "compile_time_ms", "is_generated"]].dropna()
for label, grp in plot_df.groupby("is_generated"):
    tag = "Generated" if label else "Authored"
    ax.scatter(grp["code_lines"], grp["compile_time_ms"], alpha=0.3, s=8, label=tag)
ax.set_xlabel("Lines of code")
ax.set_ylabel("Compile time (ms)")
ax.set_title("Code Lines vs Compile Time")
ax.legend()

# Scatter: preprocessed_bytes vs compile_time_ms
ax = axes[1]
plot_df = fm[["preprocessed_bytes", "compile_time_ms"]].dropna()
ax.scatter(plot_df["preprocessed_bytes"], plot_df["compile_time_ms"], alpha=0.3, s=8, color="green")
ax.set_xscale("log")
ax.set_xlabel("Preprocessed bytes")
ax.set_ylabel("Compile time (ms)")
ax.set_title("Preprocessed Size vs Compile Time")

fig.tight_layout()
plt.show()

In [ ]:
# Split distributions: generated vs authored files
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
gen_mask = fm["is_generated"] == True

for ax, col, title in zip(
    axes,
    ["compile_time_ms", "code_lines", "preprocessed_bytes"],
    ["Compile Time (ms)", "Code Lines", "Preprocessed Bytes"],
):
    for mask, label, color in [(~gen_mask, "Authored", "steelblue"), (gen_mask, "Generated", "coral")]:
        vals = fm.loc[mask, col].dropna()
        if len(vals) > 0:
            ax.hist(vals[vals > 0], bins=50, alpha=0.6, label=f"{label} (n={len(vals):,})", color=color, edgecolor="white", linewidth=0.3)
    if col in ("compile_time_ms", "preprocessed_bytes"):
        ax.set_xscale("log")
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
    ax.set_title(f"{title}: Generated vs Authored")
    ax.legend()

fig.tight_layout()
plt.show()

## 2. Target-Level Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 2a. Total build time
ax = axes[0, 0]
bt = tm["total_build_time_ms"].dropna()
ax.hist(bt[bt > 0], bins=60, edgecolor="white", linewidth=0.3)
ax.set_xscale("log")
ax.set_xlabel("Total build time (ms)")
ax.set_ylabel("Count")
ax.set_title("Target Build Time Distribution")

# 2b. File count
ax = axes[0, 1]
ax.hist(tm["file_count"], bins=60, edgecolor="white", linewidth=0.3, color="green")
ax.set_xlabel("File count")
ax.set_ylabel("Count")
ax.set_title("Files per Target")

# 2c. Code lines total
ax = axes[1, 0]
clt = tm["code_lines_total"]
ax.hist(clt[clt > 0], bins=60, edgecolor="white", linewidth=0.3, color="orange")
ax.set_xlabel("Total lines of code")
ax.set_ylabel("Count")
ax.set_title("Code Lines per Target")

# 2d. Target type distribution
ax = axes[1, 1]
type_counts = tm["target_type"].value_counts()
type_counts.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Count")
ax.set_ylabel("Target type")
ax.set_title("Target Type Distribution")
for i, v in enumerate(type_counts.values):
    ax.text(v + 0.5, i, str(v), va="center")

fig.tight_layout()
plt.show()

In [ ]:
# Box plots of compile time by target type
plot_df = tm[["target_type", "total_build_time_ms"]].dropna()
# Order by median build time
order = plot_df.groupby("target_type")["total_build_time_ms"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=plot_df, x="total_build_time_ms", y="target_type", order=order, ax=ax)
ax.set_xscale("log")
ax.set_xlabel("Total build time (ms)")
ax.set_ylabel("Target type")
ax.set_title("Build Time by Target Type")
fig.tight_layout()
plt.show()

## 3. 80/20 Analysis

In [ ]:
def pareto_analysis(df: pd.DataFrame, col: str, label: str) -> tuple[float, pd.DataFrame]:
    """Compute 80/20 Pareto analysis. Returns the % of targets for 80% of the metric."""
    sorted_df = df[["cmake_target", col]].dropna().sort_values(col, ascending=False).reset_index(drop=True)
    total = sorted_df[col].sum()
    sorted_df["cumulative"] = sorted_df[col].cumsum()
    sorted_df["cumulative_pct"] = 100 * sorted_df["cumulative"] / total
    sorted_df["target_pct"] = 100 * (sorted_df.index + 1) / len(sorted_df)

    # Find where cumulative reaches 80%
    idx_80 = (sorted_df["cumulative_pct"] >= 80).idxmax()
    pct_targets_for_80 = sorted_df.loc[idx_80, "target_pct"]

    print(f"{label}: {pct_targets_for_80:.1f}% of targets account for 80% of {label}")
    return pct_targets_for_80, sorted_df


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, label) in zip(axes, [
    ("total_build_time_ms", "Build Time"),
    ("preprocessed_bytes_total", "Preprocessed Bytes"),
    ("code_lines_total", "Code Lines"),
]):
    pct, pareto_df = pareto_analysis(tm, col, label)
    ax.plot(pareto_df["target_pct"], pareto_df["cumulative_pct"], linewidth=2)
    ax.axhline(80, color="red", linestyle="--", alpha=0.7, label="80% line")
    ax.axvline(pct, color="red", linestyle=":", alpha=0.7)
    ax.plot([0, 100], [0, 100], color="grey", linestyle="--", alpha=0.4, label="Equality line")
    ax.annotate(f"{pct:.0f}% of targets\n= 80% of {label}",
                xy=(pct, 80), xytext=(pct + 10, 60),
                arrowprops=dict(arrowstyle="->", color="red"),
                fontsize=9, color="red")
    ax.set_xlabel("Cumulative % of targets")
    ax.set_ylabel(f"Cumulative % of {label}")
    ax.set_title(f"Pareto: {label}")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 105)

fig.tight_layout()
plt.show()

In [ ]:
# Top 20 targets by build time
top20 = (
    tm[["cmake_target", "target_type", "total_build_time_ms", "file_count", "code_lines_total"]]
    .dropna(subset=["total_build_time_ms"])
    .sort_values("total_build_time_ms", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20.index = top20.index + 1
top20["build_time_s"] = (top20["total_build_time_ms"] / 1000).round(1)
total_build = tm["total_build_time_ms"].sum()
top20["pct_of_total"] = (100 * top20["total_build_time_ms"] / total_build).round(1)
top20["cumulative_pct"] = top20["pct_of_total"].cumsum().round(1)

print("TOP 20 TARGETS BY BUILD TIME")
print("=" * 100)
display_cols = ["cmake_target", "target_type", "build_time_s", "pct_of_total", "cumulative_pct", "file_count", "code_lines_total"]
print(top20[display_cols].to_string())

## 4. Codegen Analysis

In [ ]:
# Fraction of targets with generated code
n_with_codegen = (tm["codegen_ratio"] > 0).sum()
n_targets = len(tm)
print(f"Targets with generated code: {n_with_codegen}/{n_targets} ({100*n_with_codegen/n_targets:.1f}%)")
print()

# Distribution of codegen_ratio
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
cg = tm["codegen_ratio"]
ax.hist(cg[cg > 0], bins=40, edgecolor="white", linewidth=0.3, color="coral")
ax.set_xlabel("Codegen ratio")
ax.set_ylabel("Count")
ax.set_title("Codegen Ratio Distribution (targets with codegen > 0)")

# Compile time per SLOC: generated vs authored
ax = axes[1]
authored = fm.loc[~fm["is_generated"], ["compile_time_ms", "code_lines"]].dropna()
generated = fm.loc[fm["is_generated"], ["compile_time_ms", "code_lines"]].dropna()

authored_rate = authored["compile_time_ms"] / authored["code_lines"].clip(lower=1)
generated_rate = generated["compile_time_ms"] / generated["code_lines"].clip(lower=1)

data_to_plot = []
labels_to_plot = []
if len(authored_rate) > 0:
    data_to_plot.append(authored_rate)
    labels_to_plot.append(f"Authored (n={len(authored_rate):,})")
if len(generated_rate) > 0:
    data_to_plot.append(generated_rate)
    labels_to_plot.append(f"Generated (n={len(generated_rate):,})")

if data_to_plot:
    ax.boxplot(data_to_plot, labels=labels_to_plot, vert=True)
    ax.set_ylabel("Compile time per SLOC (ms/line)")
    ax.set_title("Compile Efficiency: Generated vs Authored")

fig.tight_layout()
plt.show()

# Summary stats
if len(authored_rate) > 0:
    print(f"Authored — median compile rate: {authored_rate.median():.2f} ms/line")
if len(generated_rate) > 0:
    print(f"Generated — median compile rate: {generated_rate.median():.2f} ms/line")

In [ ]:
# Which generators produce the most code? (infer from file paths)
gen_files = fm.loc[fm["is_generated"], ["source_file", "code_lines", "compile_time_ms"]].copy()

if len(gen_files) > 0:
    # Try to infer generator from path components
    def infer_generator(path: str) -> str:
        p = path.lower()
        for keyword in ["protobuf", "proto", "grpc", "thrift", "flatbuffers", "moc_", "ui_", "qrc_",
                        "generated", "autogen", "codegen", "_gen.", ".gen.", "_generated"]:
            if keyword in p:
                return keyword.strip("_.").title()
        return "Unknown"

    gen_files["generator"] = gen_files["source_file"].apply(infer_generator)
    gen_summary = (
        gen_files.groupby("generator")
        .agg(file_count=("source_file", "count"),
             total_lines=("code_lines", "sum"),
             total_compile_ms=("compile_time_ms", "sum"))
        .sort_values("total_lines", ascending=False)
    )
    print("CODE GENERATORS (inferred from file paths)")
    print("=" * 60)
    print(gen_summary.to_string())
else:
    print("No generated files found.")

## 5. GCC Phase Breakdown

In [ ]:
# Aggregate GCC phase breakdown
gcc_cols = ["gcc_parse_time_ms", "gcc_template_instantiation_ms", "gcc_codegen_time_ms", "gcc_optimization_time_ms"]
gcc_labels = ["Parsing", "Template Instantiation", "Code Generation", "Optimisation"]

gcc_data = fm[gcc_cols].dropna()
if len(gcc_data) > 0:
    totals = gcc_data.sum()
    grand_total = totals.sum()
    pcts = 100 * totals / grand_total

    print("AGGREGATE GCC PHASE BREAKDOWN")
    print("=" * 50)
    for label, col in zip(gcc_labels, gcc_cols):
        print(f"  {label:<30} {totals[col]:>12,.0f} ms  ({pcts[col]:>5.1f}%)")
    print(f"  {'Total':<30} {grand_total:>12,.0f} ms")

    # Pie chart
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.pie(totals.values, labels=gcc_labels, autopct="%1.1f%%", startangle=90)
    ax.set_title("Aggregate GCC Phase Breakdown")
    plt.show()
else:
    print("No GCC phase data available.")

In [ ]:
# Box plots of GCC phases by target (normalised to % of total)
target_gcc_cols = ["gcc_parse_time_sum_ms", "gcc_template_time_sum_ms", "gcc_codegen_phase_sum_ms", "gcc_optimization_time_sum_ms"]
target_gcc_labels = ["Parsing", "Template Inst.", "Code Gen.", "Optimisation"]

gcc_target = tm[target_gcc_cols].copy()
gcc_target_total = gcc_target.sum(axis=1)
gcc_target_pct = gcc_target.div(gcc_target_total.clip(lower=1), axis=0) * 100
gcc_target_pct.columns = target_gcc_labels

# Only include targets with non-zero GCC data
gcc_target_pct = gcc_target_pct[gcc_target_total > 0]

if len(gcc_target_pct) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    gcc_target_pct.boxplot(ax=ax)
    ax.set_ylabel("% of total GCC time")
    ax.set_title("GCC Phase Breakdown by Target (normalised)")
    fig.tight_layout()
    plt.show()

    # Targets where template instantiation dominates (>50%)
    template_heavy = gcc_target_pct[gcc_target_pct["Template Inst."] > 50]
    if len(template_heavy) > 0:
        heavy_targets = tm.loc[template_heavy.index, "cmake_target"].tolist()
        print(f"\nTargets with template instantiation > 50% of compile time ({len(heavy_targets)} targets):")
        for t in heavy_targets[:20]:
            idx = tm[tm["cmake_target"] == t].index[0]
            pct = gcc_target_pct.loc[idx, "Template Inst."]
            print(f"  {t}: {pct:.1f}%")
        if len(heavy_targets) > 20:
            print(f"  ... and {len(heavy_targets) - 20} more")
    else:
        print("\nNo targets with template instantiation > 50%.")
else:
    print("No GCC phase data available at target level.")

## 6. Correlation Analysis

In [ ]:
corr_cols = [
    "total_build_time_ms",
    "code_lines_total",
    "preprocessed_bytes_total",
    "file_count",
    "expansion_ratio_mean",
    "object_size_total_bytes",
    "total_dependency_count",
    "header_depth_max",
    "git_churn_total",
]

# Filter to available columns
available_corr_cols = [c for c in corr_cols if c in tm.columns]
corr_data = tm[available_corr_cols].dropna()

print(f"Computing Spearman correlations on {len(corr_data)} targets x {len(available_corr_cols)} metrics")

corr_matrix = corr_data.corr(method="spearman")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
# Shorten labels for readability
short_labels = [c.replace("_total", "").replace("_ms", "").replace("_bytes", "").replace("_mean", "_avg") for c in available_corr_cols]
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            xticklabels=short_labels, yticklabels=short_labels,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Spearman Correlation Matrix (Target Metrics)")
fig.tight_layout()
plt.show()

In [ ]:
# Top 5 most correlated pairs and surprising non-correlations
import itertools

pairs = []
for i, j in itertools.combinations(range(len(available_corr_cols)), 2):
    c1, c2 = available_corr_cols[i], available_corr_cols[j]
    rho = corr_matrix.loc[c1, c2]
    pairs.append((c1, c2, rho, abs(rho)))

pairs.sort(key=lambda x: x[3], reverse=True)

print("TOP 5 MOST CORRELATED PAIRS (Spearman)")
print("=" * 70)
for c1, c2, rho, _ in pairs[:5]:
    print(f"  {c1} <-> {c2}: rho = {rho:.3f}")

print("\nSURPRISING NON-CORRELATIONS (|rho| < 0.3 between metrics you'd expect to correlate)")
print("=" * 70)
# Flag pairs with low correlation that involve build time
for c1, c2, rho, abs_rho in pairs:
    if abs_rho < 0.3 and ("build_time" in c1 or "build_time" in c2):
        print(f"  {c1} <-> {c2}: rho = {rho:.3f}")

## 7. Target Segmentation

In [ ]:
# Select features for clustering
cluster_cols = [
    "total_build_time_ms",
    "code_lines_total",
    "preprocessed_bytes_total",
    "file_count",
    "total_dependency_count",
    "git_churn_total",
    "codegen_ratio",
]

available_cluster_cols = [c for c in cluster_cols if c in tm.columns]
cluster_df = tm[["cmake_target"] + available_cluster_cols].dropna().copy()
print(f"Clustering {len(cluster_df)} targets with {len(available_cluster_cols)} features")

# Standardise
scaler = StandardScaler()
X = scaler.fit_transform(cluster_df[available_cluster_cols])

# Choose k via silhouette score
k_range = range(3, min(11, len(cluster_df)))
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append((k, score))
    print(f"  k={k}: silhouette={score:.3f}")

best_k = max(silhouette_scores, key=lambda x: x[1])[0]
print(f"\nBest k = {best_k}")

In [ ]:
# Silhouette score plot
fig, ax = plt.subplots(figsize=(8, 4))
ks = [s[0] for s in silhouette_scores]
scores = [s[1] for s in silhouette_scores]
ax.plot(ks, scores, marker="o")
ax.axvline(best_k, color="red", linestyle="--", alpha=0.7, label=f"Best k={best_k}")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("K-Means: Silhouette Score vs k")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Run k-means with best k
km_final = KMeans(n_clusters=best_k, n_init=10, random_state=42)
cluster_df["cluster"] = km_final.fit_predict(X)

# Characterise each segment
segment_stats = cluster_df.groupby("cluster")[available_cluster_cols].agg(["mean", "median", "count"])

# Build a cleaner summary
summary_rows = []
for c in range(best_k):
    mask = cluster_df["cluster"] == c
    seg = cluster_df.loc[mask]
    # Get target types for this cluster
    seg_targets = seg["cmake_target"].values
    seg_types = tm.loc[tm["cmake_target"].isin(seg_targets), "target_type"].value_counts()
    dominant_type = seg_types.index[0] if len(seg_types) > 0 else "mixed"

    row = {
        "cluster": c,
        "count": len(seg),
        "dominant_type": dominant_type,
    }
    for col in available_cluster_cols:
        row[f"{col}_mean"] = seg[col].mean()
    summary_rows.append(row)

segment_summary = pd.DataFrame(summary_rows)

print("SEGMENT CHARACTERISATION")
print("=" * 100)
print(segment_summary.to_string(index=False))

In [ ]:
# Name each segment descriptively
segment_names = {}
for _, row in segment_summary.iterrows():
    c = int(row["cluster"])
    build_time = row.get("total_build_time_ms_mean", 0)
    code_lines = row.get("code_lines_total_mean", 0)
    churn = row.get("git_churn_total_mean", 0)
    codegen = row.get("codegen_ratio_mean", 0)
    count = int(row["count"])
    dom_type = row["dominant_type"]

    # Derive a descriptive name
    parts = []

    # Size
    median_lines = segment_summary["code_lines_total_mean"].median()
    if code_lines > median_lines * 2:
        parts.append("Large")
    elif code_lines < median_lines * 0.5:
        parts.append("Small")
    else:
        parts.append("Medium")

    # Churn
    median_churn = segment_summary["git_churn_total_mean"].median()
    if churn > median_churn * 2:
        parts.append("churning")
    elif churn < median_churn * 0.5:
        parts.append("stable")

    # Codegen
    if codegen > 0.5:
        parts.append("codegen-heavy")

    # Type
    type_short = dom_type.replace("_", " ")
    parts.append(type_short + "s")

    name = " ".join(parts)
    segment_names[c] = name
    print(f"Cluster {c} (n={count}): \"{name}\"")
    print(f"  Build time: {build_time:,.0f} ms avg | Code: {code_lines:,.0f} lines avg | Churn: {churn:,.0f} avg | Codegen: {codegen:.1%}")
    print()

cluster_df["segment_name"] = cluster_df["cluster"].map(segment_names)

## 8. Persist Output

In [ ]:
# Build target_features DataFrame: original metrics + cluster assignment + derived metrics
target_features = tm.copy()

# Merge cluster assignments
cluster_assignments = cluster_df[["cmake_target", "cluster", "segment_name"]].copy()
target_features = target_features.merge(cluster_assignments, on="cmake_target", how="left")

# Add derived metrics
target_features["compile_time_per_file_ms"] = (
    target_features["compile_time_sum_ms"] / target_features["file_count"].clip(lower=1)
)
target_features["compile_time_per_kloc_ms"] = (
    target_features["compile_time_sum_ms"] / (target_features["code_lines_total"].clip(lower=1) / 1000)
)
target_features["bytes_per_line"] = (
    target_features["preprocessed_bytes_total"] / target_features["code_lines_total"].clip(lower=1)
)

# Save
out_path = INTERMEDIATE_DIR / "target_features.parquet"
target_features.to_parquet(out_path, index=False)

print(f"Saved target_features.parquet: {target_features.shape[0]} rows x {target_features.shape[1]} cols")
print(f"  -> {out_path}")
print(f"  Columns added: cluster, segment_name, compile_time_per_file_ms, compile_time_per_kloc_ms, bytes_per_line")